# Negative-binomial re-analysis: egress exits vs. wildfire fatalities

Reruns the count model from [egress-fatalities-reanalysis.md](egress-fatalities-reanalysis.md) on the
corrected dataset (`data/fire-fatality-corrected.csv`, 38 fire × census-place records).

**Model:** `deaths ~ exits + offset(log population)`, negative binomial (NB2, α estimated by ML).
The offset turns the count model into a *rate* model, and NB handles the heavy overdispersion that
makes Poisson (and OLS on per-capita ratios) inappropriate here.

Sections: data → NB fit (original vs corrected denominators) → fitted rate curve → the paper's
Fig S4 OLS for comparison → leave-one-out jackknife.

In [1]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.simplefilter("ignore", RuntimeWarning)  # statsmodels NB llf eval emits benign log(0) warnings

df = pd.read_csv("data/fire-fatality-corrected.csv")
df["label"] = df.Fire.str.title() + " / " + df.census_place
print(f"{len(df)} records, {df.census_place.nunique()} unique places, {df.Identified_fatalities.sum()} deaths")
df.sort_values("Identified_fatalities", ascending=False).head(10)

38 records, 35 unique places, 312 deaths


,Fire,census_place,Identified_fatalities,exits,population_orig,population_corrected,fatality_pct_corrected,label
0,Hawaii/Lahaina,Lahiaina,102,4.0,13261,13261,0.7692,Hawaii/Lahaina / Lahiaina
1,CAMP,Paradise,66,6.0,7730,26800,0.2463,Camp / Paradise
2,Eaton,Altadena,19,20.0,41921,41921,0.0453,Eaton / Altadena
3,NORTH COMPLEX,Berry Creek,13,3.0,1190,1190,1.0924,North Complex / Berry Creek
4,2016 Great Smoky Mountain wildfires,Gatlinburg,12,9.0,3684,3684,0.3257,2016 Great Smoky Mountain Wildfires / Gatlinburg
5,TUBBS,Larkfield-Wikiup CDP,11,11.0,7595,7595,0.1448,Tubbs / Larkfield-Wikiup CDP
6,CAMP,Concow,9,6.0,306,710,1.2676,Camp / Concow
7,CAMP,Magalia,9,10.0,10537,10537,0.0854,Camp / Magalia
8,REDWOOD VALLEY,Redwood Valley,9,12.0,1798,1798,0.5006,Redwood Valley / Redwood Valley
9,Palisades,Malibu,5,14.5,10915,10915,0.0458,Palisades / Malibu


## 1. Negative binomial fit — original vs. corrected denominators

Six populations in the source data were post-fire/depopulated census figures or geographically
misassigned, all at low-exit places, all inflating the fatality rate toward the paper's conclusion.
We fit the NB model with both the original and corrected denominators.

In [2]:
def fit_nb(pop_col):
    return smf.negativebinomial(
        "Identified_fatalities ~ exits", data=df, offset=np.log(df[pop_col])
    ).fit(disp=0)

fits = {label: fit_nb(col) for label, col in
        [("original", "population_orig"), ("corrected", "population_corrected")]}

rows = []
for label, m in fits.items():
    b, (lo, hi) = m.params["exits"], m.conf_int().loc["exits"]
    rows.append({
        "denominators": label,
        "coef (exits)": round(b, 4),
        "IRR / exit": round(np.exp(b), 3),
        "effect / exit": f"{(np.exp(b) - 1) * 100:+.1f}%",
        "95% CI (IRR)": f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]",
        "p": f"{m.pvalues['exits']:.1e}",
        "alpha": round(m.params["alpha"], 3),
    })
pd.DataFrame(rows).set_index("denominators")

,coef (exits),IRR / exit,effect / exit,95% CI (IRR),p,alpha
denominators,,,,,,
original,-0.2075,0.813,-18.7%,"[0.754, 0.875]",4.8e-08,1.127
corrected,-0.1785,0.837,-16.3%,"[0.777, 0.900]",1.9e-06,1.047


In [3]:
# Why NB and not Poisson: Pearson dispersion under Poisson is ~12x what equidispersion allows
pois = smf.glm("Identified_fatalities ~ exits", data=df, family=sm.families.Poisson(),
               offset=np.log(df.population_corrected)).fit()
print(f"Poisson Pearson dispersion phi = {pois.pearson_chi2 / pois.df_resid:.1f}  (equidispersion -> 1)")

Poisson Pearson dispersion phi = 12.3  (equidispersion -> 1)


## 2. Fitted rate curve

Observed per-capita fatality rate (corrected denominators, log scale) with the NB-implied rate
`exp(b0 + b1·exits)`. The decline is smooth and constant-per-exit — **no threshold/kink at ~6 exits**.
Marker area ∝ population; hover for fire/place.

In [4]:
m = fits["corrected"]
df["rate_pct"] = df.Identified_fatalities / df.population_corrected * 100

fig = px.scatter(
    df, x="exits", y="rate_pct", size="population_corrected", hover_name="label",
    hover_data={"Identified_fatalities": True, "population_corrected": ":,",
                "rate_pct": ":.3f", "exits": True},
    size_max=40, log_y=True, opacity=0.75,
    labels={"exits": "egress exits", "rate_pct": "fatalities per 100 residents (log)"},
)
xs = np.linspace(df.exits.min(), df.exits.max(), 100)
pred = np.exp(m.params["Intercept"] + m.params["exits"] * xs) * 100
fig.add_trace(go.Scatter(x=xs, y=pred, mode="lines", name="NB fitted rate",
                         line=dict(color="firebrick", width=2)))
fig.update_layout(title="Wildfire fatality rate vs egress exits (corrected denominators)",
                  height=480, legend=dict(orientation="h", y=1.08))
fig.show()

## 3. The paper's Fig S4 spec (OLS on per-capita rate), for comparison

OLS on a noisy ratio is the wrong model here, but it's what the paper reported (p = 0.048, our
unweighted rerun p = 0.036). On corrected denominators it slips below conventional significance.

In [5]:
rows = []
for label, col in [("original", "population_orig"), ("corrected", "population_corrected")]:
    rate = df.Identified_fatalities / df[col] * 100
    ols = smf.ols("rate ~ exits", data=df.assign(rate=rate)).fit()
    rows.append({"denominators": label, "slope": round(ols.params["exits"], 4),
                 "p": round(ols.pvalues["exits"], 4), "R^2": round(ols.rsquared, 3)})
pd.DataFrame(rows).set_index("denominators")

,slope,p,R^2
denominators,,,
original,-0.0534,0.0362,0.116
corrected,-0.0386,0.0601,0.095


## 4. Leave-one-out jackknife

Refit the corrected-denominator NB model 38 times, dropping one record each time. If any single
community were driving the result, its deletion would move the coefficient toward zero.

In [6]:
jk = []
for i in df.index:
    sub = df.drop(i)
    mi = smf.negativebinomial("Identified_fatalities ~ exits", data=sub,
                              offset=np.log(sub.population_corrected)).fit(disp=0)
    jk.append({"dropped": df.loc[i, "label"], "coef": mi.params["exits"]})
jk = pd.DataFrame(jk).sort_values("coef").reset_index(drop=True)
full_coef = fits["corrected"].params["exits"]
print(f"full-model coef = {full_coef:.4f}; LOO range [{jk.coef.min():.4f}, {jk.coef.max():.4f}]; "
      f"all negative: {(jk.coef < 0).all()}")

fig = px.scatter(jk, x="coef", y="dropped", height=720,
                 labels={"coef": "exits coefficient after dropping this record", "dropped": ""})
fig.add_vline(x=full_coef, line_dash="dash", line_color="firebrick",
              annotation_text="full model", annotation_position="top")
fig.add_vline(x=0, line_color="gray")
fig.update_layout(title="Jackknife: NB exits coefficient, leave-one-out",
                  yaxis=dict(tickfont=dict(size=9)))
fig.show()

full-model coef = -0.1785; LOO range [-0.1960, -0.1679]; all negative: True


## 5. Robustness

Six checks, in roughly increasing order of severity. The headline asymptotic p-value (~2×10⁻⁶) deserves
skepticism with n = 38, a heavy-tailed exits distribution, and selection on fatal events — the honest
question is whether the *effect* survives, and at what strength.

**5a. Specification battery.** Same mean model, different likelihoods. The zero-truncated NB matters
most: the dataset only contains places with ≥1 death, so the untruncated likelihood is mildly
misspecified by construction.

In [7]:
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

df["lpop"] = np.log(df.population_corrected)
y, X = df.Identified_fatalities, sm.add_constant(df[["exits"]])

specs = {}
specs["NB2 ML (baseline)"] = (m.params["exits"], m.pvalues["exits"])
pois_hc3 = smf.glm("Identified_fatalities ~ exits", df, family=sm.families.Poisson(),
                   offset=df.lpop).fit(cov_type="HC3")
specs["Poisson, HC3 robust SE"] = (pois_hc3.params["exits"], pois_hc3.pvalues["exits"])
glmnb = smf.glm("Identified_fatalities ~ exits", df,
                family=sm.families.NegativeBinomial(alpha=m.params["alpha"]), offset=df.lpop).fit()
specs["GLM-NB (alpha fixed at ML)"] = (glmnb.params["exits"], glmnb.pvalues["exits"])
ztnb = TruncatedLFNegativeBinomialP(y, X, offset=df.lpop, truncation=0).fit(disp=0, maxiter=500)
specs["Zero-truncated NB (selection-honest)"] = (ztnb.params["exits"], ztnb.pvalues["exits"])

pd.DataFrame(
    [{"specification": k, "IRR / exit": round(np.exp(b), 3),
      "effect / exit": f"{(np.exp(b) - 1) * 100:+.1f}%", "p": f"{p:.1e}"}
     for k, (b, p) in specs.items()]
).set_index("specification")

,IRR / exit,effect / exit,p
specification,,,
NB2 ML (baseline),0.837,-16.3%,1.9e-06
"Poisson, HC3 robust SE",0.863,-13.7%,1.4e-04
GLM-NB (alpha fixed at ML),0.837,-16.3%,1.1e-07
Zero-truncated NB (selection-honest),0.852,-14.8%,2.7e-02


**5b. Functional form — is there any threshold?** If the paper's "~6 exits" breakpoint were real,
a hinge term at 6 should beat the smooth linear model. It doesn't: AIC is a statistical tie and the
hinge term is not significant.

In [8]:
df["hinge6"] = np.maximum(df.exits - 6, 0)
forms = {
    "linear (baseline)": m,
    "log(exits)": smf.negativebinomial("Identified_fatalities ~ np.log(exits)", df, offset=df.lpop).fit(disp=0),
    "quadratic": smf.negativebinomial("Identified_fatalities ~ exits + I(exits**2)", df, offset=df.lpop).fit(disp=0),
    "hinge at 6 exits": smf.negativebinomial("Identified_fatalities ~ exits + hinge6", df, offset=df.lpop).fit(disp=0),
}
print(pd.DataFrame([{"form": k, "AIC": round(v.aic, 2), "loglik": round(v.llf, 2)} for k, v in forms.items()])
      .set_index("form").to_string())
print(f"\nhinge term:     coef = {forms['hinge at 6 exits'].params['hinge6']:+.3f}, "
      f"p = {forms['hinge at 6 exits'].pvalues['hinge6']:.3f}")
print(f"quadratic term: p = {forms['quadratic'].pvalues['I(exits ** 2)']:.3f}")

                      AIC  loglik
form                             
linear (baseline)  228.42 -111.21
log(exits)         233.99 -113.99
quadratic          228.98 -110.49
hinge at 6 exits   228.36 -110.18

hinge term:     coef = -0.310, p = 0.155
quadratic term: p = 0.230


**5c. Offset relaxation.** The offset forces deaths ∝ population (elasticity 1). Freeing it tests
whether that assumption manufactures the effect. The fitted elasticity is ~0.6 (significantly < 1) and
the exits effect attenuates to −9.4%/exit but stays significant — some of the baseline effect rides on
the proportionality assumption, the direction doesn't.

In [9]:
free = smf.negativebinomial("Identified_fatalities ~ exits + lpop", df).fit(disp=0)
t = free.t_test("lpop = 1")
b, p = free.params["exits"], free.pvalues["exits"]
print(f"exits:    IRR = {np.exp(b):.3f} ({(np.exp(b)-1)*100:+.1f}%/exit), p = {p:.4f}")
print(f"log(pop): elasticity = {free.params['lpop']:.3f}, H0 elasticity=1: p = {t.pvalue.item():.4f}")

exits:    IRR = 0.906 (-9.4%/exit), p = 0.0112
log(pop): elasticity = 0.601, H0 elasticity=1: p = 0.0004


**5d. Block deletions.** The jackknife (§4) drops one record at a time; here we delete whole
suspect blocks: the two mass-casualty fires (Lahaina + Paradise are 168 of 312 deaths), all six
corrected-denominator rows, and the tiny CDPs whose populations are barely benchmarkable.

In [10]:
def refit(sub):
    mm = smf.negativebinomial("Identified_fatalities ~ exits", sub,
                              offset=np.log(sub.population_corrected)).fit(disp=0)
    return mm.params["exits"], mm.pvalues["exits"]

blocks = {
    "full data (baseline)": df,
    "drop Lahaina + Paradise (168/312 deaths)": df[~df.census_place.isin(["Lahiaina", "Paradise"])],
    "drop the 6 corrected-denominator rows": df[df.population_orig == df.population_corrected],
    "drop tiny CDPs (pop < 500)": df[df.population_corrected >= 500],
}
pd.DataFrame(
    [{"subset": k, "n": len(v), "IRR / exit": round(np.exp(refit(v)[0]), 3),
      "effect / exit": f"{(np.exp(refit(v)[0]) - 1) * 100:+.1f}%", "p": f"{refit(v)[1]:.1e}"}
     for k, v in blocks.items()]
).set_index("subset")

,n,IRR / exit,effect / exit,p
subset,,,,
full data (baseline),38,0.837,-16.3%,1.9e-06
drop Lahaina + Paradise (168/312 deaths),36,0.838,-16.2%,6.4e-06
drop the 6 corrected-denominator rows,32,0.829,-17.1%,6.3e-07
drop tiny CDPs (pop < 500),30,0.867,-13.3%,2.3e-04


**5e. Cluster bootstrap by fire + permutation test.** Records cluster within fires (the Camp fire
alone contributes three places), so we resample *fires* with replacement. And because asymptotic
p-values are optimistic at n = 38, we also build the exact null by permuting exits. **This is the most
important check in the notebook:** the permutation p is ~0.04 — three orders of magnitude weaker than
the asymptotic 2×10⁻⁶, though still under 0.05.

In [11]:
from plotly.subplots import make_subplots

rng = np.random.default_rng(42)
fires = df.Fire.unique()
boot = []
for _ in range(1000):
    bs = pd.concat([df[df.Fire == f] for f in rng.choice(fires, size=len(fires), replace=True)])
    try:
        bm = smf.negativebinomial("Identified_fatalities ~ exits", bs,
                                  offset=np.log(bs.population_corrected)).fit(disp=0, maxiter=200)
        if bm.mle_retvals.get("converged"):
            boot.append(bm.params["exits"])
    except Exception:
        pass
boot = np.array(boot)

perm = []
for _ in range(2000):
    d2 = df.assign(pex=rng.permutation(df.exits.values))
    try:
        pm = smf.negativebinomial("Identified_fatalities ~ pex", d2, offset=df.lpop).fit(disp=0, maxiter=200)
        perm.append(pm.params["pex"])
    except Exception:
        pass
perm = np.array(perm)

b_obs = m.params["exits"]
ci = np.exp(np.percentile(boot, [2.5, 97.5]))
p_perm = (np.abs(perm) >= abs(b_obs)).mean()
print(f"cluster bootstrap ({len(boot)} converged): IRR 95% CI [{ci[0]:.3f}, {ci[1]:.3f}], "
      f"P(coef >= 0) = {(boot >= 0).mean():.3f}")
print(f"permutation test ({len(perm)} fits):       two-sided p = {p_perm:.4f}  "
      f"(asymptotic p = {m.pvalues['exits']:.1e})")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Cluster bootstrap (by fire): IRR per exit", "Permutation null vs observed coefficient"))
fig.add_trace(go.Histogram(x=np.exp(boot), nbinsx=60, marker_color="steelblue", showlegend=False), 1, 1)
fig.add_vline(x=1.0, line_color="gray", row=1, col=1)
fig.add_vline(x=float(np.exp(b_obs)), line_dash="dash", line_color="firebrick",
              annotation_text="observed", row=1, col=1)
fig.add_trace(go.Histogram(x=perm, nbinsx=60, marker_color="darkseagreen", showlegend=False), 1, 2)
fig.add_vline(x=float(b_obs), line_dash="dash", line_color="firebrick",
              annotation_text="observed", row=1, col=2)
fig.update_layout(height=420, title="Resampling-based inference")
fig.show()

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian fail

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian fail

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian fail

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian fail

/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likeli

cluster bootstrap (967 converged): IRR 95% CI [0.785, 0.921], P(coef >= 0) = 0.001
permutation test (2000 fits):       two-sided p = 0.0395  (asymptotic p = 1.9e-06)


/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/Users/michael/projects/egress-fatalities/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likeli

## Conclusions

- **The effect direction is real and robust:** ~16% lower fatality rate per additional egress road
  (IRR 0.837, 95% CI [0.777, 0.900], p ≈ 2×10⁻⁶) on corrected denominators; ~19% on the original ones.
- **No threshold:** the decline is smooth; the paper's "~6 exits" breakpoint was an artifact of
  regressing a reverse cumulative sum on its own sort key.
- **The paper's per-capita OLS does not survive the denominator corrections** (p 0.036 → 0.060).
- **No single community drives the result** — the jackknife coefficient stays in [−0.196, −0.168],
  and the most influential point (Eaton/Altadena) *strengthens* the effect when dropped.

**Robustness (§5):**

- **The effect direction survives every check** — alternative likelihoods (including zero-truncated NB:
  −14.8%/exit, p = 0.027), block deletions (dropping Lahaina+Paradise, the corrected rows, or tiny
  CDPs barely moves it), a cluster bootstrap by fire (IRR 95% CI [0.785, 0.921]), and a free
  population elasticity (attenuates to −9.4%/exit, p = 0.011, still negative).
- **But the headline p-value should be read as ~0.04, not 10⁻⁶.** The permutation test — the honest
  small-sample null — gives p ≈ 0.04. Significant, not overwhelming.
- **Still no threshold:** a hinge at 6 exits is an AIC tie with linear and its term is not
  significant (p = 0.155).
- Still observational, ~35 communities, selection on fatal events, exits confounded with community
  type — **no causal or policy reading is warranted.**